# HDB Resale Flat Prices ETL Pipeline

**Production-Ready Orchestrator**

This notebook orchestrates a complete Extract-Transform-Validate-Load pipeline for HDB resale flat price data.
Each cell represents a distinct stage with clear input/output metrics.

---
## STAGE 0: INITIALIZATION

Setup environment, imports, and pipeline components

In [1]:
import sys
import os
from datetime import datetime

# Add project root to path for imports
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("="*80)
print("HDB ETL PIPELINE - INITIALIZATION".center(80))
print("="*80)
print(f"\nProject Root: {project_root}")
print(f"Python Version: {sys.version.split()[0]}")
print(f"Execution Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

                       HDB ETL PIPELINE - INITIALIZATION                        

Project Root: g:\My Drive\Work and Finances\HDB\etl_project
Python Version: 3.11.5
Execution Time: 2026-03-03 21:08:36


In [2]:
from pyspark.sql import SparkSession

# Import ETL Components
from config.etl_config import ETLConfig
from extractors.data_extractor import DataExtractor
import transformers.data_transformer as data_transformer_module
from validators.data_validator import DataValidator
import loaders.data_loader as data_loader_module
from utils.logger import setup_logging, get_logger

DataTransformer = data_transformer_module.DataTransformer
DataLoader = data_loader_module.DataLoader

# Setup logging
setup_logging("INFO")
logger = get_logger(__name__)

print("\n✓ All dependencies imported successfully")
print("✓ DataTransformer module loaded")
print("✓ DataLoader module loaded")


✓ All dependencies imported successfully
✓ DataTransformer module loaded
✓ DataLoader module loaded


In [3]:
# Initialize Spark
spark = SparkSession.builder \
    .appName("HDB_ResalePrices_ETL") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("\n✓ Spark Session Initialized")
print(f"  App Name: {spark.sparkContext.appName}")
print(f"  Master: {spark.sparkContext.master}")
print(f"  Spark Version: {spark.version}")


✓ Spark Session Initialized
  App Name: HDB_ResalePrices_ETL
  Master: local[*]
  Spark Version: 3.5.2


In [4]:
# Initialize Configuration & Components
config = ETLConfig()
extractor = DataExtractor(spark)
transformer = DataTransformer()
validator = DataValidator()
loader = DataLoader(spark)

print("\n✓ ETL Configuration & Components Initialized")
print(f"\n  CONFIG PARAMETERS:")
print(f"    Source Path: {config.source_path}")
print(f"    Destination Path: {config.destination_path}")
print(f"    Source Format: {config.source_format}")
print(f"    Target Format: {config.target_format}")
print(f"    Min Rows (Validation): {config.min_rows}")
print(f"\n  CRITICAL COLUMNS:")
for i, col_name in enumerate(config.critical_columns, 1):
    print(f"    {i:2d}. {col_name}")


✓ ETL Configuration & Components Initialized

  CONFIG PARAMETERS:
    Source Path: ../ResaleFlatPrices/
    Destination Path: output/processed_data
    Source Format: csv
    Target Format: csv
    Min Rows (Validation): 100

  CRITICAL COLUMNS:
     1. month
     2. town
     3. flat_type
     4. block
     5. street_name
     6. storey_range
     7. floor_area_sqm
     8. flat_model
     9. lease_commence_date
    10. resale_price


---
## STAGE 1: EXTRACT - Load Raw Data

Read all source CSV files and combine into single dataset

In [5]:
# EXTRACT: Load all CSV files and combine
combined_df, extraction_stats = extractor.extract_and_combine(config, loader)

# Save initial extracted DataFrame for later comparison
combined_df_initial = combined_df

# Print extraction summary
print(f"\n{'-'*80}")
print(f"EXTRACTION SUMMARY".center(80))
print(f"{'-'*80}")
for stat in extraction_stats:
    print(f"  {stat['status']} | {stat['file']:50s} | {stat['rows']:>8,d} rows")

if combined_df is not None:
    total_rows = combined_df.count()
    total_cols = len(combined_df.columns)
    rows_before_filter = total_rows
    print(f"\n✓ COMBINED DATASET:")
    print(f"    Total Records: {total_rows:,}")
    print(f"    Total Columns: {total_cols}")
    print(f"    Columns: {combined_df.columns}")
else:
    print(f"\n✗ No data extracted")
    rows_before_filter = 0

2026-03-03 21:08:39,061 - extractors.data_extractor - INFO - ================================================================================
2026-03-03 21:08:39,061 - extractors.data_extractor - INFO - EXTRACTION: Discovering and loading CSV files
2026-03-03 21:08:39,061 - extractors.data_extractor - INFO - ================================================================================
2026-03-03 21:08:39,063 - extractors.data_extractor - INFO - Found 5 CSV files in ../ResaleFlatPrices/
2026-03-03 21:08:39,065 - extractors.data_extractor - INFO - 
  [1/5] Extracting: Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv
2026-03-03 21:08:39,065 - extractors.data_extractor - INFO - Reading CSV: ../ResaleFlatPrices/Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv
2026-03-03 21:08:42,798 - extractors.data_extractor - INFO -     Records: 287,196 | Columns: 10
2026-03-03 21:08:42,798 - loaders.data_loader - INFO - Loading CSV to output/raw\Resale Flat Prices (Based on Ap


--------------------------------------------------------------------------------
                               EXTRACTION SUMMARY                               
--------------------------------------------------------------------------------
  ✓ Extracted | Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv |  287,196 rows
  ✓ Extracted | Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv |  369,651 rows
  ✓ Extracted | Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv |   37,153 rows
  ✓ Extracted | Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv |   52,203 rows
  ✓ Extracted | Resale flat prices based on registration date from Jan-2017 onwards.csv |  225,608 rows

✓ COMBINED DATASET:
    Total Records: 971,811
    Total Columns: 10
    Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']


---
## STAGE 2: FILTER - Apply Date Range Filter

Filter records to match business requirements (2012-01 to 2016-12)

In [6]:
print("\n" + "="*80)
print("FILTERING STAGE - Date Range Filter".center(80))
print("="*80)

start_month = config.start_month
end_month = config.end_month

print(f"\nFILTER PARAMETERS:")
print(f"  Date Column: month")
print(f"  Start Date: {start_month}")
print(f"  End Date: {end_month}")
print(f"  Records Before: {rows_before_filter:,}")

# Apply date range filter using transformer
combined_df = transformer.filter_by_date_range(
    combined_df,
    date_column="month",
    start_date=start_month,
    end_date=end_month
)

rows_after_filter = combined_df.count()

print(f"\nFILTER RESULTS:")
print(f"  Records Passed: {rows_after_filter:,}")
if rows_before_filter > 0:
    print(f"  Pass Rate: {(rows_after_filter/rows_before_filter*100):.2f}%")

2026-03-03 21:08:49,094 - transformers.data_transformer - INFO - Filtering by date range: 2012-01 to 2016-12



                      FILTERING STAGE - Date Range Filter                       

FILTER PARAMETERS:
  Date Column: month
  Start Date: 2012-01
  End Date: 2016-12
  Records Before: 971,811


2026-03-03 21:08:49,814 - transformers.data_transformer - INFO - ✓ Date range filtering completed:
2026-03-03 21:08:49,814 - transformers.data_transformer - INFO -   Total records: 971,811
2026-03-03 21:08:49,814 - transformers.data_transformer - INFO -   Passed filter: 92,544
2026-03-03 21:08:49,814 - transformers.data_transformer - INFO -   Failed filter: 879,267



FILTER RESULTS:
  Records Passed: 92,544
  Pass Rate: 9.52%


---
## STAGE 3: PROFILE - Analyze Raw Data

Comprehensive data profiling before transformation

In [7]:
print("\n" + "="*80)
print("DATA PROFILING STAGE (Before Transformation)".center(80))
print("="*80)

profile = validator.profile_data(combined_df)

print(f"\nPROFILE SUMMARY:")
print(f"  Total Rows: {profile['row_count']:,}")
print(f"  Total Columns: {profile['column_count']}")
print(f"  Null Values Found: {len(profile['null_analysis'])} columns have nulls")

# Display Categorical Columns
if profile['categorical_columns']:
    print(f"\n3. CATEGORICAL COLUMNS ANALYSIS")
    print(f"  {'-'*76}")
    for col_name, distinct_count in profile['categorical_columns'].items():
        print(f"   {col_name:30s}: {distinct_count:>8,} distinct values")

# Display Numeric Statistics (from profile_data method)
if profile['numeric_statistics']:
    print(f"\n4. NUMERIC COLUMNS STATISTICS")
    print(f"  {'-'*76}")
    
    if 'resale_price' in profile['numeric_statistics']:
        stats = profile['numeric_statistics']['resale_price']
        print(f"\n  RESALE PRICE:")
        print(f"    Min:    ${stats['min']:>15,.0f}")
        print(f"    Max:    ${stats['max']:>15,.0f}")
        print(f"    Median: ${stats['median']:>15,.0f}")
        print(f"    Mean:   ${stats['mean']:>15,.2f}")
    
    if 'floor_area_sqm' in profile['numeric_statistics']:
        stats = profile['numeric_statistics']['floor_area_sqm']
        print(f"\n  FLOOR AREA (sqm):")
        print(f"    Min:    {stats['min']:>15,.2f}")
        print(f"    Max:    {stats['max']:>15,.2f}")
        print(f"    Median: {stats['median']:>15,.2f}")
        print(f"    Mean:   {stats['mean']:>15,.2f}")



2026-03-03 21:08:50,109 - validators.data_validator - INFO - ================================================================================
2026-03-03 21:08:50,109 - validators.data_validator - INFO -                                  DATA PROFILING                                 
2026-03-03 21:08:50,109 - validators.data_validator - INFO - ================================================================================



                  DATA PROFILING STAGE (Before Transformation)                  


2026-03-03 21:08:50,404 - validators.data_validator - INFO - 
1. BASIC DATASET INFORMATION
2026-03-03 21:08:50,406 - validators.data_validator - INFO - --------------------------------------------------------------------------------
2026-03-03 21:08:50,406 - validators.data_validator - INFO -    Total Rows: 92,544
2026-03-03 21:08:50,406 - validators.data_validator - INFO -    Total Columns: 10
2026-03-03 21:08:50,407 - validators.data_validator - INFO - 
   Column Names and Types:
2026-03-03 21:08:50,407 - validators.data_validator - INFO -     1. month                          -> timestamp      
2026-03-03 21:08:50,407 - validators.data_validator - INFO -     2. town                           -> string         
2026-03-03 21:08:50,408 - validators.data_validator - INFO -     3. flat_type                      -> string         
2026-03-03 21:08:50,408 - validators.data_validator - INFO -     4. block                          -> string         
2026-03-03 21:08:50,408 - validators.data


PROFILE SUMMARY:
  Total Rows: 92,544
  Total Columns: 10
  Null Values Found: 0 columns have nulls

3. CATEGORICAL COLUMNS ANALYSIS
  ----------------------------------------------------------------------------
   town                          :       26 distinct values
   flat_type                     :        7 distinct values
   block                         :    2,139 distinct values
   street_name                   :      522 distinct values
   storey_range                  :       25 distinct values
   flat_model                    :       20 distinct values

4. NUMERIC COLUMNS STATISTICS
  ----------------------------------------------------------------------------

  RESALE PRICE:
    Min:    $        190,000
    Max:    $      1,150,000
    Median: $        428,000
    Mean:   $     450,938.97

  FLOOR AREA (sqm):
    Min:              31.00
    Max:             280.00
    Median:           95.00
    Mean:             96.57


---
## STAGE 4a: CLEANING - Type Cast, Clean, Standardize, Outlier Removal, Deduplicate

Apply core data cleaning operations before transformation/enrichment

In [8]:
from pyspark.sql.functions import col
combined_df.filter(col("resale_price").isNotNull()).show()

+-------------------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------+
|              month|      town|flat_type|block|      street_name|storey_range|floor_area_sqm|    flat_model|lease_commence_date|resale_price|
+-------------------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------+
|2012-01-01 00:00:00|ANG MO KIO|   2 ROOM|  406|ANG MO KIO AVE 10|    01 TO 03|          44.0|      Improved|               1979|    257800.0|
|2012-01-01 00:00:00|ANG MO KIO|   2 ROOM|  314| ANG MO KIO AVE 3|    07 TO 09|          44.0|      Improved|               1978|    263000.0|
|2012-01-01 00:00:00|ANG MO KIO|   2 ROOM|  314| ANG MO KIO AVE 3|    10 TO 12|          44.0|      Improved|               1978|    275000.0|
|2012-01-01 00:00:00|ANG MO KIO|   2 ROOM|  170| ANG MO KIO AVE 4|    01 TO 03|          45.0|      Improved|               1986|    260000.0|

In [9]:
print("\n" + "="*80)
print("CLEANING STAGE".center(80))
print("="*80)

rows_before_cleaning = combined_df.count()
cols_before_cleaning = len(combined_df.columns)

print(f"\nBEFORE CLEANING:")
print(f"  Rows: {rows_before_cleaning:,}")
print(f"  Columns: {cols_before_cleaning}")

print(f"\nAPPLYING CLEANING STEPS:")

# 1. Type Casting
print(f"  [1/5] Type Casting...")
combined_df = transformer.cast_datatypes(combined_df)

# 2. Deduplicating by full rows
print(f"  [2/5] Deduplicating by full rows...")
combined_df = transformer.deduplicate_by_full_rows(combined_df)

# 3. Standardization
print(f"  [3/5] Data Standardization...")
combined_df = transformer.standardize_data(combined_df)

# 4. Remove Resale Price Outliers (IQR)
print(f"  [4/5] Removing Resale Price Outliers...")
combined_df = transformer.remove_resale_price_outliers(combined_df, iqr_multiplier=3.0)

# 5. Deduplicate (keep highest resale_price per composite key)
print(f"  [5/5] Deduplicating Records...")
rows_before_dedup = combined_df.count()
key_columns = [c for c in combined_df.columns if c != "resale_price"]
combined_df = transformer.deduplicate_by_composite_key(
    combined_df,
    key_columns=key_columns
)
rows_after_dedup = combined_df.count()

# Snapshot output for Cleaned folder
cleaned_df = combined_df

rows_after_cleaning = cleaned_df.count()
cols_after_cleaning = len(cleaned_df.columns)

print(f"\nAFTER CLEANING:")
print(f"  Rows: {rows_after_cleaning:,}")
print(f"  Columns: {cols_after_cleaning}")
print(f"  New Columns Added: {cols_after_cleaning - cols_before_cleaning}")
print(f"  Rows Removed: {rows_before_cleaning - rows_after_cleaning:,}")
print(f"  Deduplicated Rows Removed: {rows_before_dedup - rows_after_dedup:,}")


                                 CLEANING STAGE                                 


2026-03-03 21:08:54,827 - transformers.data_transformer - INFO - Casting data types...
2026-03-03 21:08:54,871 - transformers.data_transformer - INFO - ✓ Data types casted
2026-03-03 21:08:54,871 - transformers.data_transformer - INFO - Deduplicating by full rows...



BEFORE CLEANING:
  Rows: 92,544
  Columns: 10

APPLYING CLEANING STEPS:
  [1/5] Type Casting...
  [2/5] Deduplicating by full rows...


2026-03-03 21:08:55,958 - transformers.data_transformer - INFO - ✓ Deduplicated data: 92,544 → 92,271 rows
2026-03-03 21:08:55,959 - transformers.data_transformer - INFO -   Full-row duplicates removed: 273
2026-03-03 21:08:55,959 - transformers.data_transformer - INFO - Standardizing data...
2026-03-03 21:08:56,025 - transformers.data_transformer - INFO - ✓ Data standardized


  [3/5] Data Standardization...
  [4/5] Removing Resale Price Outliers...


2026-03-03 21:08:59,595 - transformers.data_transformer - INFO - ✓ Outlier removal complete:
2026-03-03 21:08:59,596 - transformers.data_transformer - INFO -   Grouping columns: ['town', 'flat_type', 'floor_area_sqm', 'flat_model', 'lease_commence_date']
2026-03-03 21:08:59,596 - transformers.data_transformer - INFO -   IQR multiplier: 3.0
2026-03-03 21:08:59,597 - transformers.data_transformer - INFO -   Rows removed: 240


  [5/5] Deduplicating Records...


2026-03-03 21:09:01,231 - transformers.data_transformer - INFO - Deduplicating by composite key...
2026-03-03 21:09:04,735 - transformers.data_transformer - INFO - ✓ Deduplication complete:
2026-03-03 21:09:04,735 - transformers.data_transformer - INFO -   Original Records: 92,031
2026-03-03 21:09:04,736 - transformers.data_transformer - INFO -   Deduplicated Records (Kept): 90,705
2026-03-03 21:09:04,736 - transformers.data_transformer - INFO -   Records Removed: 1,326



AFTER CLEANING:
  Rows: 90,705
  Columns: 10
  New Columns Added: 0
  Rows Removed: 1,839
  Deduplicated Rows Removed: 1,326


---
## STAGE 4b: TRANSFORM - Lease and Identifier Enrichment

Apply remaining transformation/enrichment steps after cleaning

In [10]:
print("\n" + "="*80)
print("TRANSFORMATION STAGE".center(80))
print("="*80)

rows_before_transform = combined_df.count()
cols_before_transform = len(combined_df.columns)

print(f"\nBEFORE TRANSFORMATION:")
print(f"  Rows: {rows_before_transform:,}")
print(f"  Columns: {cols_before_transform}")

print(f"\nAPPLYING TRANSFORMATIONS:")

# 1. Lease Calculation
print(f"  [1/2] Computing Remaining Lease...")
combined_df = transformer.compute_remaining_lease(combined_df)

# 2. Create Resale Identifier
print(f"  [2/2] Creating Resale Identifier...")
combined_df = transformer.create_resale_identifier(combined_df)

# Snapshot output for Transformed folder (pre-hash)
transformed_df = combined_df

rows_after_transform = transformed_df.count()
cols_after_transform = len(transformed_df.columns)

print(f"\nAFTER TRANSFORMATION:")
print(f"  Rows: {rows_after_transform:,}")
print(f"  Columns: {cols_after_transform}")
print(f"  New Columns Added: {cols_after_transform - cols_before_transform}")
print(f"  Rows Removed: {rows_before_transform - rows_after_transform:,}")


                              TRANSFORMATION STAGE                              


2026-03-03 21:09:09,563 - transformers.data_transformer - INFO - Computing remaining lease duration...
2026-03-03 21:09:09,590 - transformers.data_transformer - INFO - ✓ Lease duration computed
2026-03-03 21:09:09,591 - transformers.data_transformer - INFO - Creating Resale Identifier...
2026-03-03 21:09:09,623 - transformers.data_transformer - INFO - ✓ Resale Identifier created



BEFORE TRANSFORMATION:
  Rows: 90,705
  Columns: 10

APPLYING TRANSFORMATIONS:
  [1/2] Computing Remaining Lease...
  [2/2] Creating Resale Identifier...

AFTER TRANSFORMATION:
  Rows: 90,705
  Columns: 13
  New Columns Added: 3
  Rows Removed: 0


---
## STAGE 4c: HASH - Hash Resale Identifier

Apply hash transformation on the identifier and prepare hashed dataset output

In [12]:
print("\n" + "="*80)
print("HASH STAGE".center(80))
print("="*80)

rows_before_hash = combined_df.count()
cols_before_hash = len(combined_df.columns)

print(f"\nBEFORE HASHING:")
print(f"  Rows: {rows_before_hash:,}")
print(f"  Columns: {cols_before_hash}")

print(f"\nAPPLYING HASH TRANSFORMATION:")
print(f"  [1/1] Hashing Resale Identifier...")
combined_df = transformer.hash_resale_identifier(combined_df)

# Snapshot output for Hashed folder
hashed_df = combined_df

rows_after_hash = hashed_df.count()
cols_after_hash = len(hashed_df.columns)

print(f"\nAFTER HASHING:")
print(f"  Rows: {rows_after_hash:,}")
print(f"  Columns: {cols_after_hash}")
print(f"  New Columns Added: {cols_after_hash - cols_before_hash}")
print(f"  Rows Removed: {rows_before_hash - rows_after_hash:,}")


                                   HASH STAGE                                   


2026-03-03 21:10:11,537 - transformers.data_transformer - INFO - Hashing Resale Identifier with SHA-256...



BEFORE HASHING:
  Rows: 90,705
  Columns: 13

APPLYING HASH TRANSFORMATION:
  [1/1] Hashing Resale Identifier...


2026-03-03 21:10:15,504 - transformers.data_transformer - INFO - ✓ Resale Identifier hashed with SHA-256



AFTER HASHING:
  Rows: 90,705
  Columns: 13
  New Columns Added: 0
  Rows Removed: 0


---
## STAGE 5: VALIDATE - Data Quality Checks

Perform comprehensive validation before loading

In [14]:
print("\n" + "="*80)
print("VALIDATION STAGE".center(80))
print("="*80)

print(f"\nVALIDATION PARAMETERS:")
print(f"  Minimum Row Count: {config.min_rows:,}")
print(f"  Critical Columns: {len(config.critical_columns)}")

# Run validations
is_valid, validation_results = validator.validate(
    combined_df,
    config.critical_columns,
    min_rows=config.min_rows
)

print(f"\nVALIDATION RESULTS:")
for key, value in validation_results.items():
    print(f"  {key}: {value}")

print(f"\n{'='*80}")
if is_valid:
    print("✓✓✓ ALL VALIDATIONS PASSED ✓✓✓".center(80))
    print("Pipeline proceeding to LOAD stage".center(80))
else:
    print("✗✗✗ VALIDATION FAILED ✗✗✗".center(80))
    print("Pipeline will ABORT load operation".center(80))
print(f"{'='*80}")

2026-03-03 21:28:41,723 - validators.data_validator - INFO - ==================================================
2026-03-03 21:28:41,723 - validators.data_validator - INFO - Starting Data Validation
2026-03-03 21:28:41,723 - validators.data_validator - INFO - ==================================================
2026-03-03 21:28:41,725 - validators.data_validator - INFO - Checking row count...



                                VALIDATION STAGE                                

VALIDATION PARAMETERS:
  Minimum Row Count: 100
  Critical Columns: 10


2026-03-03 21:28:44,004 - validators.data_validator - INFO - ✓ Row count valid: 90705
2026-03-03 21:28:44,005 - validators.data_validator - INFO - Checking for null values...
2026-03-03 21:28:46,450 - validators.data_validator - INFO - ✓ No null values in critical columns
2026-03-03 21:29:11,726 - validators.data_validator - INFO - ✓ All validations passed!



VALIDATION RESULTS:
  row_count: 90705
  null_check: {'month': 0, 'town': 0, 'flat_type': 0, 'block': 0, 'street_name': 0, 'storey_range': 0, 'floor_area_sqm': 0, 'flat_model': 0, 'lease_commence_date': 0, 'resale_price': 0}
  master_fields_check: True
  master_fields_report: {'month': {'min_observed': '2012-01-01', 'max_observed': '2016-12-01', 'null_count': 0, 'pct_null': 0.0, 'out_of_expected_range_count': 0, 'expected_range': '2012-01 to 2016-12', 'pass': True}, 'town': {'distinct_count': 26, 'top_values': [('JURONG WEST', 7408), ('WOODLANDS', 7274), ('TAMPINES', 6630), ('BEDOK', 5910), ('YISHUN', 5811), ('SENGKANG', 5768), ('HOUGANG', 4667), ('ANG MO KIO', 4454), ('CHOA CHU KANG', 3882), ('BUKIT BATOK', 3712)], 'null_count': 0, 'pct_null': 0.0, 'freq_threshold_count': 90, 'pass': True}, 'flat_type': {'distinct_count': 7, 'top_values': [('4 ROOM', 35963), ('3 ROOM', 25531), ('5 ROOM', 21037), ('EXECUTIVE', 7175), ('2 ROOM', 924), ('1 ROOM', 48), ('MULTI-GENERATION', 27)], 'null_co

---
## STAGE 6: LOAD - Write Processed Data

Load clean, validated data to destination (only if validation passed)

In [15]:
print("\n" + "="*80)
print("LOAD STAGE".center(80))
print("="*80)

if is_valid:
    try:
        print(f"\nLOAD PARAMETERS:")
        print(f"  Base Destination: {config.destination_path}")
        print(f"  Format: {config.target_format}")
        print(f"  Records to Load: {combined_df.count():,}")
        print(f"  Write Mode: overwrite")

        # Required output folders
        cleaned_path = f"{config.destination_path}/Cleaned"
        transformed_path = f"{config.destination_path}/Transformed"
        hashed_path = f"{config.destination_path}/Hashed"

        print(f"\nWriting CLEANED dataset...")
        loader.load(cleaned_df, cleaned_path, config.target_format, mode="overwrite", output_file_name="cleaned_df")

        print(f"Writing TRANSFORMED dataset...")
        loader.load(transformed_df, transformed_path, config.target_format, mode="overwrite", output_file_name="transformed_df")

        print(f"Writing HASHED dataset...")
        loader.load(hashed_df, hashed_path, config.target_format, mode="overwrite", output_file_name="hashed_df")

        print(f"\n✓ DATA LOAD SUCCESSFUL")
        print(f"  Cleaned Path:     {cleaned_path}")
        print(f"  Transformed Path: {transformed_path}")
        print(f"  Hashed Path:      {hashed_path}")
        print(f"  Records Loaded: {combined_df.count():,}")

        load_successful = True

    except Exception as e:
        logger.error(f"Load failed: {e}")
        print(f"\n✗ LOAD FAILED")
        print(f"  Error: {e}")
        load_successful = False
else:
    print(f"\n⚠ LOAD SKIPPED")
    print(f"  Reason: Validation failed")
    print(f"  Data was NOT written to destination")
    load_successful = False


                                   LOAD STAGE                                   

LOAD PARAMETERS:
  Base Destination: output/processed_data
  Format: csv


2026-03-03 21:29:26,595 - loaders.data_loader - INFO - Loading data to output/processed_data/Cleaned in csv format
2026-03-03 21:29:26,595 - loaders.data_loader - INFO - Loading CSV to output/processed_data/Cleaned


  Records to Load: 90,705
  Write Mode: overwrite

Writing CLEANED dataset...


2026-03-03 21:29:28,791 - loaders.data_loader - INFO - Renamed output file to cleaned_df.csv
2026-03-03 21:29:28,795 - loaders.data_loader - INFO - ✓ CSV load complete
2026-03-03 21:29:30,067 - loaders.data_loader - INFO - ✓ Data load complete: 90,705 records
2026-03-03 21:29:30,068 - loaders.data_loader - INFO - Loading data to output/processed_data/Transformed in csv format
2026-03-03 21:29:30,069 - loaders.data_loader - INFO - Loading CSV to output/processed_data/Transformed


Writing TRANSFORMED dataset...


2026-03-03 21:29:32,339 - loaders.data_loader - INFO - Renamed output file to transformed_df.csv
2026-03-03 21:29:32,339 - loaders.data_loader - INFO - ✓ CSV load complete
2026-03-03 21:29:33,899 - loaders.data_loader - INFO - ✓ Data load complete: 90,705 records
2026-03-03 21:29:33,900 - loaders.data_loader - INFO - Loading data to output/processed_data/Hashed in csv format
2026-03-03 21:29:33,900 - loaders.data_loader - INFO - Loading CSV to output/processed_data/Hashed


Writing HASHED dataset...


2026-03-03 21:29:36,367 - loaders.data_loader - INFO - Renamed output file to hashed_df.csv
2026-03-03 21:29:36,367 - loaders.data_loader - INFO - ✓ CSV load complete
2026-03-03 21:29:37,980 - loaders.data_loader - INFO - ✓ Data load complete: 90,705 records



✓ DATA LOAD SUCCESSFUL
  Cleaned Path:     output/processed_data/Cleaned
  Transformed Path: output/processed_data/Transformed
  Hashed Path:      output/processed_data/Hashed
  Records Loaded: 90,705


In [16]:
# Calculate failed_df: records in initial extract but not in final output
# Use composite key to identify records that made it through
composite_key = ["town", "block", "street_name", "flat_type", "month"]

print("\n" + "="*80)
print("CALCULATING FAILED RECORDS".center(80))
print("="*80)

print(f"\nInitial Extracted Records: {combined_df_initial.count():,}")
print(f"Final Output Records: {combined_df.count():,}")

# Left anti-join: records in initial that don't exist in final
failed_df = combined_df_initial.join(
    combined_df.select(*composite_key).distinct(),
    on=composite_key,
    how="left_anti"
)

failed_count = failed_df.count()
success_count = combined_df.count()

print(f"\nFailed Records (removed during pipeline): {failed_count:,}")
print(f"Success Rate: {(success_count/(combined_df_initial.count())*100):.2f}%")

# Write required FAILED dataset output folder
failed_path = f"{config.destination_path}/Failed"
print(f"\nWriting FAILED dataset...")
loader.load(failed_df, failed_path, config.target_format, mode="overwrite", output_file_name="failed_df")
print(f"✓ Failed dataset written: {failed_path}")


                           CALCULATING FAILED RECORDS                           

Initial Extracted Records: 971,811
Final Output Records: 90,705

Failed Records (removed during pipeline): 879,485


2026-03-03 21:30:47,420 - loaders.data_loader - INFO - Loading data to output/processed_data/Failed in csv format
2026-03-03 21:30:47,421 - loaders.data_loader - INFO - Loading CSV to output/processed_data/Failed


Success Rate: 9.33%

Writing FAILED dataset...


2026-03-03 21:30:53,235 - loaders.data_loader - INFO - Renamed output file to failed_df.csv
2026-03-03 21:30:53,237 - loaders.data_loader - INFO - ✓ CSV load complete
2026-03-03 21:30:57,371 - loaders.data_loader - INFO - ✓ Data load complete: 879,485 records


✓ Failed dataset written: output/processed_data/Failed


---
## CLEANUP & RESOURCES

In [17]:
print("\nCleaning up resources...")
spark.stop()
print("✓ Spark session stopped")
print("✓ All resources released")


Cleaning up resources...
✓ Spark session stopped
✓ All resources released
